# 03 — Interpretable logistic model

This is an explanatory classification exercise on a historical snapshot, not a production forecast. Target fields and churn reasons are excluded.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'clean_customers.csv')
df = df[df['customer_status'] != 'Joined'].copy()
numeric = ['age', 'num_dependents', 'num_referrals', 'tenure_months', 'monthly_charge', 'bundle_count']
categorical = ['gender', 'contract', 'internet_type', 'payment_method', 'offer']
X = df[numeric + categorical]
y = df['churn'].astype(int)
print('rows:', len(X), '| churn rate:', f'{y.mean():.2%}')

In [ ]:
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])
model = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])
minority_count = int(y.value_counts().min())
n_splits = min(5, minority_count)
if n_splits < 2:
    raise ValueError('At least two examples from each class are required.')
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scores = cross_validate(model, X, y, cv=cv, scoring=['roc_auc', 'precision', 'recall'])
metrics = pd.DataFrame({
    'metric': ['ROC-AUC', 'Precision', 'Recall'],
    'mean': [scores['test_roc_auc'].mean(), scores['test_precision'].mean(), scores['test_recall'].mean()],
    'std': [scores['test_roc_auc'].std(), scores['test_precision'].std(), scores['test_recall'].std()],
})
metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(metrics['metric'], metrics['mean'], yerr=metrics['std'], color=['#4472C4', '#70AD47', '#D64550'], capsize=5)
ax.set_ylim(0, 1)
ax.set_ylabel('Cross-validation score')
ax.set_title(f'Logistic regression ({n_splits}-fold CV)')
fig.tight_layout()
fig.savefig(ROOT / 'reports' / 'figures' / 'logistic_cv.png', dpi=150, bbox_inches='tight')
plt.show()